# MAMMAL Embedding Extraction

Turn each molecule into one **768-number vector** with MAMMAL's encoder, to train LightGBM on
(see [MAMMAL_embedding_lgbm_cv.ipynb](MAMMAL_embedding_lgbm_cv.ipynb)).

Only the **encoder** half of the model runs: the molecule's prompt is read, which gives one vector per token,
and those are averaged over the real tokens (the padding is skipped). This is the same recipe MAMMAL's own
vLLM plugin uses. The decoder, which would write `<0>` or `<1>`, is not used at all.

The embeddings can come from the **pretrained** model (`MODEL_PATH = None`, no fine-tuning needed, the cheapest
way to get a first result) or from a model fine-tuned by
[MAMMAL_finetuning.ipynb](MAMMAL_finetuning.ipynb).

**Environment:** run with the `biomed-mammal` kernel.

**Input:** a `.csv` or `.parquet` file with a SMILES column. Molecules RDKit cannot read are skipped and
listed in `invalid_smiles.csv`.

**Output**, in `output/<DATASET>/embeddings/<EMBEDDING_NAME>/`, the same layout as the MMELON embeddings so
the LightGBM notebooks read it unchanged:
```
mammal/part-00000.parquet   # row_index, smiles, embedding (768 numbers), one file per chunk
invalid_smiles.csv          # only if some SMILES could not be read
config.json                 # settings used
```
Chunks are written as they finish, so re-running the notebook continues where it stopped.

## 1. Inputs and settings

In [ ]:
from pathlib import Path

# Dataset name: all outputs go under aircheck/output/<DATASET>/
DATASET = "sample-data"

# Name of this extraction; outputs go to output/<DATASET>/embeddings/<EMBEDDING_NAME>/
EMBEDDING_NAME = "train-pretrained"

# None = the pretrained base model; or a fine-tuned best_model/ folder from MAMMAL_finetuning.ipynb
MODEL_PATH = None

# Molecules
DATA_PATH = Path("../output") / DATASET / "cv_folds" / "train_cluster_cv_data.csv"
# For the final test set (LightGBM inference): EMBEDDING_NAME = "test-pretrained",
# DATA_PATH = Path("../../../biomed-multi-view/aircheck/sample_data/test.parquet")
SMILES_COL = "SMILES"

# The folder the embeddings are written to. "mammal" is the only view this model has; the name matches
# the "text"/"image"/"graph"/"fused" folders of the MMELON embeddings.
VIEW = "mammal"

In [ ]:
import os

BASE_MODEL = "ibm/biomed.omics.bl.sm.ma-ted-458m"  # used when MODEL_PATH is None

CHUNK_SIZE = 1000         # molecules per saved part file
BATCH_SIZE = 32
NUM_WORKERS = min(24, os.cpu_count())

# Prompt lengths, in tokens; keep these the same as the fine-tuning run
MAX_SMILES_TOKENS = 300
MAX_PROMPT_TOKENS = 320

In [ ]:
import json

import torch


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "mammal").is_dir() and (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError(f"Run this notebook from inside the biomed-multi-alignment repo (cwd={start})")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
OUTPUT_DIR = REPO_ROOT / "aircheck" / "output"
OUT_DIR = OUTPUT_DIR / DATASET / "embeddings" / EMBEDDING_NAME
assert Path(DATA_PATH).exists(), f"Data not found: {DATA_PATH}"

if MODEL_PATH is None:
    MODEL_SOURCE = BASE_MODEL
else:
    MODEL_SOURCE = Path(MODEL_PATH).expanduser().resolve()
    if not (MODEL_SOURCE / "config.json").exists() and (MODEL_SOURCE / "best_model").is_dir():
        MODEL_SOURCE = MODEL_SOURCE / "best_model"      # a run folder was given
    assert MODEL_SOURCE.is_dir(), f"Model not found: {MODEL_SOURCE}"
    MODEL_SOURCE = str(MODEL_SOURCE)
TOKENIZER_SOURCE = MODEL_SOURCE

OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / VIEW).mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

config = dict(model=str(MODEL_SOURCE), pretrained=MODEL_PATH is None, data_path=str(DATA_PATH),
              smiles_col=SMILES_COL, view=VIEW, chunk_size=CHUNK_SIZE,
              max_smiles_tokens=MAX_SMILES_TOKENS, max_prompt_tokens=MAX_PROMPT_TOKENS)
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=2))

print(f"Model  : {MODEL_SOURCE}{' (pretrained)' if MODEL_PATH is None else ' (fine-tuned)'}")
print(f"Data   : {DATA_PATH}")
print(f"Output : {OUT_DIR}")
print(f"Device : {DEVICE}")

## 2. Prepare data

In [ ]:
import numpy as np
import pandas as pd
import torch
from functools import partial
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from sklearn.metrics import average_precision_score, roc_auc_score

from fuse.data.utils.collates import CollateDefault
from mammal.keys import *
import pyarrow.parquet as pq
def read_table(path, columns, optional=()) -> pd.DataFrame:
    # Read only the needed columns from a .csv or .parquet file; optional columns are read if present
    path = Path(path)
    is_parquet = path.suffix.lower() in (".parquet", ".pq")
    available = pq.ParquetFile(path).schema_arrow.names if is_parquet else pd.read_csv(path, nrows=0).columns.tolist()
    missing = [c for c in columns if c not in available]
    assert not missing, f"{path.name} has no column(s) {missing}; available: {available}"
    wanted = list(dict.fromkeys([*columns, *(c for c in optional if c in available)]))
    df = pd.read_parquet(path, columns=wanted) if is_parquet else pd.read_csv(path, usecols=wanted)
    return df[wanted].reset_index(drop=True)
RDLogger.DisableLog("rdApp.*")

input_df = read_table(DATA_PATH, [SMILES_COL])

smiles = input_df[SMILES_COL].astype(str).str.strip()
valid = input_df[SMILES_COL].notna() & smiles.map(lambda s: Chem.MolFromSmiles(s) is not None)

molecules = pd.DataFrame({"row_index": input_df.index[valid], "smiles": smiles[valid]}).reset_index(drop=True)
if (~valid).any():
    input_df.loc[~valid, [SMILES_COL]].rename_axis("row_index").to_csv(OUT_DIR / "invalid_smiles.csv")

n_chunks = (len(molecules) + CHUNK_SIZE - 1) // CHUNK_SIZE
print(f"Rows           : {len(input_df):,}")
print(f"Valid SMILES   : {len(molecules):,} in {n_chunks:,} chunks of up to {CHUNK_SIZE:,}")
if (~valid).any():
    print(f"Invalid SMILES : {int((~valid).sum()):,} (skipped, listed in invalid_smiles.csv)")

## 3. Load the model

Only the encoder is used, so the model runs with gradients switched off.

In [ ]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf")

from fuse.data.tokenizers.modular_tokenizer.op import ModularTokenizerOp
from mammal.model import Mammal

tokenizer_op = ModularTokenizerOp.from_pretrained(TOKENIZER_SOURCE)

NEG_TOKEN_ID = tokenizer_op.get_token_id("<0>")
POS_TOKEN_ID = tokenizer_op.get_token_id("<1>")
PAD_TOKEN_ID = tokenizer_op.get_token_id("<PAD>")
print(f"Tokenizer loaded. Answer tokens: <0> = {NEG_TOKEN_ID}, <1> = {POS_TOKEN_ID}")

In [ ]:
IGNORE_INDEX = -100
TOKEN_WEIGHTS_KEY = "data.token_weights"
MAX_ANSWER_TOKENS = 4
ANSWER_POSITION = 1  # the decoder writes <SENTINEL_ID_0> first, then the answer


def encoder_prompt(smiles: str) -> str:
    """The text MAMMAL reads for one molecule."""
    return (
        f"<@TOKENIZER-TYPE=SMILES><SENTINEL_ID_0>"
        f"<MOLECULAR_ENTITY><MOLECULAR_ENTITY_SMALL_MOLECULE>"
        f"<@TOKENIZER-TYPE=SMILES@MAX-LEN={MAX_SMILES_TOKENS}>"
        f"<SEQUENCE_NATURAL_START>{smiles}<SEQUENCE_NATURAL_END><EOS>"
    )


def prepare_sample(sample_dict: dict, tokenizer_op, pos_weight: float = 1.0, with_label: bool = True) -> dict:
    """Tokenize one sample: the prompt, and (for training) the answer and its loss weight."""
    sample_dict[ENCODER_INPUTS_STR] = encoder_prompt(sample_dict["data.smiles"])
    tokenizer_op(
        sample_dict=sample_dict,
        key_in=ENCODER_INPUTS_STR,
        key_out_tokens_ids=ENCODER_INPUTS_TOKENS,
        key_out_attention_mask=ENCODER_INPUTS_ATTENTION_MASK,
        max_seq_len=MAX_PROMPT_TOKENS,
    )
    for k in (ENCODER_INPUTS_TOKENS, ENCODER_INPUTS_ATTENTION_MASK):
        sample_dict[k] = torch.tensor(sample_dict[k])

    if not with_label:
        return sample_dict

    label = int(sample_dict["data.label"])

    # What the model should write
    sample_dict[LABELS_STR] = f"<@TOKENIZER-TYPE=SMILES><SENTINEL_ID_0><{label}><EOS>"
    tokenizer_op(
        sample_dict=sample_dict, key_in=LABELS_STR,
        key_out_tokens_ids=LABELS_TOKENS, key_out_attention_mask=LABELS_ATTENTION_MASK,
        max_seq_len=MAX_ANSWER_TOKENS,
    )
    for k in (LABELS_TOKENS, LABELS_ATTENTION_MASK):
        sample_dict[k] = torch.tensor(sample_dict[k])
    # Padding must not count towards the loss
    sample_dict[LABELS_TOKENS][sample_dict[LABELS_TOKENS] == PAD_TOKEN_ID] = IGNORE_INDEX

    # What the decoder is fed during training (teacher forcing)
    sample_dict[DECODER_INPUTS_STR] = f"<@TOKENIZER-TYPE=SMILES><DECODER_START><SENTINEL_ID_0><{label}><EOS>"
    tokenizer_op(
        sample_dict=sample_dict, key_in=DECODER_INPUTS_STR,
        key_out_tokens_ids=DECODER_INPUTS_TOKENS, key_out_attention_mask=DECODER_INPUTS_ATTENTION_MASK,
        max_seq_len=MAX_ANSWER_TOKENS,
    )
    for k in (DECODER_INPUTS_TOKENS, DECODER_INPUTS_ATTENTION_MASK):
        sample_dict[k] = torch.tensor(sample_dict[k])

    # Class balancing: weight every counted position of this sample, 0 where the loss is ignored
    weight = pos_weight if label == 1 else 1.0
    sample_dict[TOKEN_WEIGHTS_KEY] = weight * (sample_dict[LABELS_TOKENS] != IGNORE_INDEX).float()
    return sample_dict


class MoleculeDataset(Dataset):
    """One row per molecule, from a DataFrame with 'smiles' and (for training) 'label' columns."""

    def __init__(self, df, tokenizer_op, pos_weight: float = 1.0, with_label: bool = True):
        self.smiles = df["smiles"].tolist()
        self.labels = df["label"].tolist() if with_label else [0] * len(df)
        self.tokenizer_op, self.pos_weight, self.with_label = tokenizer_op, pos_weight, with_label

    def __len__(self) -> int:
        return len(self.smiles)

    def __getitem__(self, idx: int) -> dict:
        sample = {"data.sample_id": idx, "data.smiles": self.smiles[idx], "data.label": self.labels[idx]}
        return prepare_sample(sample, self.tokenizer_op, self.pos_weight, self.with_label)


def make_collate():
    """Trim the padding each batch does not need (10-20% faster)."""
    return CollateDefault(special_handlers_keys={
        ENCODER_INPUTS_TOKENS: partial(CollateDefault.crop_padding, pad_token_id=PAD_TOKEN_ID),
        ENCODER_INPUTS_ATTENTION_MASK: partial(CollateDefault.crop_padding, pad_token_id=False),
    })
model = Mammal.from_pretrained(MODEL_SOURCE).to(DEVICE).eval()
print(f"Loaded {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")


@torch.no_grad()
def embed(batch_dict) -> np.ndarray:
    """One vector per molecule: the encoder's token vectors, averaged over the real tokens."""
    mask = batch_dict[ENCODER_INPUTS_ATTENTION_MASK]
    hidden = model.t5_model.encoder(
        inputs_embeds=model._calculate_inputs_embeddings(batch_dict),
        attention_mask=mask,
    )["last_hidden_state"]                                   # (batch, tokens, 768)
    weights = mask.unsqueeze(-1).to(hidden.dtype)            # 0 for padding, 1 for real tokens
    pooled = (hidden * weights).sum(dim=1) / weights.sum(dim=1).clamp(min=1)
    return pooled.float().cpu().numpy()

## 4. Extract the embeddings

In [ ]:
%%time
import gc

from tqdm.auto import tqdm


def part_path(chunk: int) -> Path:
    return OUT_DIR / VIEW / f"part-{chunk:05d}.parquet"


for chunk in tqdm(range(n_chunks), desc="chunks"):
    if part_path(chunk).exists():
        continue  # finished in an earlier run
    chunk_df = molecules.iloc[chunk * CHUNK_SIZE:(chunk + 1) * CHUNK_SIZE]

    loader = DataLoader(
        MoleculeDataset(chunk_df, tokenizer_op, with_label=False),
        batch_size=BATCH_SIZE, shuffle=False, collate_fn=make_collate(), num_workers=NUM_WORKERS,
    )
    outputs = []
    for batch in loader:
        for k in (ENCODER_INPUTS_TOKENS, ENCODER_INPUTS_ATTENTION_MASK):
            batch[k] = batch[k].to(DEVICE)
        outputs.append(embed(batch))

    emb = np.concatenate(outputs)
    assert len(emb) == len(chunk_df), f"chunk {chunk}: {len(emb)} embeddings for {len(chunk_df)} molecules"
    part = chunk_df[["row_index", "smiles"]].assign(embedding=list(emb))
    # Write to a hidden temp file first, so an interrupted write never looks like a finished part
    final = part_path(chunk)
    tmp = final.with_name(f".{final.name}.tmp")
    part.to_parquet(tmp, index=False)
    tmp.rename(final)

    del loader, outputs
    gc.collect()

print(f"Done: {len(molecules):,} molecules, {emb.shape[1]} dimensions each")

## 5. Check the embeddings

In [ ]:
df_view = pd.read_parquet(OUT_DIR / VIEW).sort_values("row_index").reset_index(drop=True)
matrix = np.stack(df_view["embedding"].to_numpy())
assert len(df_view) == len(molecules) and df_view["row_index"].is_unique
print(f"{VIEW}: {matrix.shape[0]:,} molecules x {matrix.shape[1]} dims")
display(df_view.head())

In [ ]:
# Files produced
total = 0
for path in sorted(OUT_DIR.rglob("*")):
    if path.is_file():
        size = path.stat().st_size / 1e6
        total += size
print(f"{len(list(OUT_DIR.rglob('*.parquet')))} part files, {total:.1f} MB in {OUT_DIR}")